In [2]:
from Environment import *
from DDPG import *
from NN_Module import *
from config_56bus import Config
import os

import torch
import matplotlib.pyplot as plt
import scienceplots
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
from scipy import interpolate

from loguru import logger
import os
import numpy as np

NATURE_CONFIG = {
    "width": 1800,
    "height": 900,
    "font_base": 28,
    "font_title": 32,
    "font_axis": 24,
    "font_legend": 24,
    "dpi": 300
}

### a simple logger
logger.remove()
logger.add(sys.stderr, level='DEBUG')



4

In [ ]:
env_seed = 24        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l 11h 13l 18l 24L

agent_num = 5
agent_policy_net = []
safe_agent_net = []

### create testing environment
injection_bus = np.array([18, 21, 30, 45, 53])-1
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)
# state, topology, senario = env.reset_topo(seed=env_seed)      #change topology
state, topology, senario = env.reset(seed=env_seed)             #not change
topology = torch.cuda.FloatTensor(topology).unsqueeze(0)

### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=Config.topology_hidden_dim)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=Config.hidden_dim_56bus).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    #value_net_dict = torch.load(f'check_points/value_net/2023-06-19/Step_200_Seed_12_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-08-09/Step_250_Seed_23_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-08-15/Step_900_Seed_33_a{i}.pth')
    #policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2023-09-21/Step_900_Seed_10_a{i}.pth'))
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth'))

    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(agent_num):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/stable_ddpg/policy_net_checkpoint_a{i}.pth')

    safe_agent_net[i].load_state_dict(policy_net_dict)

episode_reward = 0
episode_control = 0
voltage = []
q = []
cost = []

last_action = np.zeros((agent_num,1))

done_record = True
state, topology, senario = env.reset_topo(seed=env_seed)      #change topology
# state, topology, senario = env.reset(seed=env_seed)             #not change
topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
for t in range(100):
    action = []
    for i in range(agent_num):
        action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
        action_agent = action_agent.detach().cpu().numpy()[0]
        action.append(action_agent)

    if np.min(action) < -1.0 or np.max(action) > 1.0:
        logger.warning('control output saturated! min is {}, max is {}', np.min(action), np.max(action))

    action = last_action -np.asarray(action)
    
    last_action = np.copy(action)
    
    try:
        next_state, reward, done = env.step(action)
    except:
        logger.error(sys.exc_info())
        logger.error('power flow not converge at {}', t)
        break

    if done and done_record:
        logger.info('RLC-FT stable at step {}', t)
        logger.info('RLC-FT stable cost is {}', episode_control)
        done_record = False

    voltage.append(state)

    q.append(action)

    state = next_state
    
    episode_reward += reward
    
    cost.append(-reward)
    
    episode_control += LA.norm(action, 2)

    # if done:
    #     break

voltage_RL = np.asarray(voltage)
q_RL =  np.asarray(q)
cost_RL =  np.asarray(cost)
logger.info('control cost of flexible controller is {}',episode_control)
logger.info('objective of flexible controller is {}', np.sum(cost_RL))

### test the base line controller
state, topology, senario = env.reset_topo(seed=env_seed)      #change topology
# state, topology, senario = env.reset(seed=env_seed)             #not change
episode_reward = 0
episode_control = 0
num_agent = 5
voltage = []
q = []
cost = []

last_action = np.zeros((num_agent,1))
done_record = True
for t in range(100):
    state1 = np.asarray(state-env.vmax)
    state2 = np.asarray(env.vmin-state)
    d_v = (np.maximum(state1, 0)-np.maximum(state2, 0)).reshape((num_agent,1))
    
    action = (last_action - 10*d_v)
    
    last_action = np.copy(action)
    
    try:
        next_state, reward, done = env.step(action)
    except:
        logger.error(sys.exc_info())
        logger.error('power flow not converge at {}', t)
        break

    if done and done_record:
        logger.info('Linear stable at step {}', t)
        logger.info('Linear stable cost is {}', episode_control)
        done_record = False

    voltage.append(state)

    q.append(action)

    state = next_state
    
    episode_reward += reward
    
    cost.append(-reward)
    
    episode_control += LA.norm(action, 2)

    # if done:
    #     break

voltage_baseline = np.asarray(voltage)
q_baseline =  np.asarray(q)
cost_baseline =  np.asarray(cost)
logger.info('control cost of linear controller is {}',episode_control)
logger.info('objective of linear controller is {}', np.sum(cost_baseline))

### test the safe policy net
state, topology, senario = env.reset_topo(seed=env_seed)      #change topology
# state, topology, senario = env.reset(seed=env_seed)             #not change
episode_reward = 0
episode_control = 0
num_agent = 5
safe_voltage = []
safe_q = []
safe_cost = []

last_action = np.zeros((num_agent,1))
done_record = True
for t in range(100):
    action = []
    for i in range(num_agent):
        action_agent = safe_agent_net[i].get_action(torch.cuda.FloatTensor([state[i]]).float().reshape(1,1))
        action.append(action_agent)

    if 5*np.min(action) < -1.0 or 5*np.max(action) > 1.0:
        logger.warning('control output saturated! min is {}, max is {}', 5*np.min(action), 5*np.max(action))
    
    action = last_action - 5*np.asarray(action).reshape((num_agent, 1))
    
    last_action = np.copy(action)
    
    try:
        next_state, reward, done = env.step(action)
    except:
        logger.error(sys.exc_info())
        logger.error('power flow not converge at {}', t)
        break

    if done and done_record:
        logger.info('Safe-DDPG stable at step {}', t)
        logger.info('Safe-DDPG stable cost is {}', episode_control)
        done_record = False

    safe_voltage.append(state)

    safe_q.append(action)

    state = next_state
    
    episode_reward += reward
    
    safe_cost.append(-reward)
    
    episode_control += LA.norm(action, 2)

    # if done:
    #     break

safe_voltage = np.asarray(safe_voltage)
safe_q =  np.asarray(safe_q)
safe_cost =  np.asarray(safe_cost)
logger.info('control cost of safe-DDPG is {}',episode_control)
logger.info('objective of linear controller is {}', np.sum(safe_cost))

2025-05-26 17:03:09.305 | DEBUG    | Environment:reset:226 - Episode start at low volatage, lowest is 0.877884762543176 pu...
2025-05-26 17:03:09.753 | INFO     | __main__:<module>:74 - RLC-FT stable at step 3
2025-05-26 17:03:09.758 | INFO     | __main__:<module>:75 - RLC-FT stable cost is 5.314494777245655
2025-05-26 17:03:13.737 | INFO     | __main__:<module>:96 - control cost of flexible controller is 242.07517839389703
2025-05-26 17:03:13.739 | INFO     | __main__:<module>:97 - objective of flexible controller is 1167.46199905179
2025-05-26 17:03:14.094 | INFO     | __main__:<module>:128 - Linear stable at step 22
2025-05-26 17:03:14.096 | INFO     | __main__:<module>:129 - Linear stable cost is 42.864776963792366
2025-05-26 17:03:15.226 | INFO     | __main__:<module>:150 - control cost of linear controller is 205.85012052644672
2025-05-26 17:03:15.241 | INFO     | __main__:<module>:151 - objective of linear controller is 1253.5982629326697
2025-05-26 17:03:15.277 | WARNING  | __m

In [40]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
import numpy as np
import os

bus_titles = ['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53']
selected_buses_indices_actual = [2,3,4] # Indices for Bus 21, Bus 30, Bus 53
selected_bus_titles_display = [bus_titles[i] for i in selected_buses_indices_actual]

bus_colors = pc.qualitative.Plotly[:len(selected_buses_indices_actual)]

method_styles_base = { # Renamed to avoid conflict, will be used to get dash style
    'RLC-FT': dict(dash='solid', width=5),
    'Safe-DDPG': dict(dash='dashdot', width=2.5),
    'Linear': dict(dash='dash', width=2.5)
}

# Method colors FOR COST PLOT (Panel b) AND FOR PROMINENT AVERAGE LINES IN PANEL A
method_primary_colors = { # Renamed from method_colors_cost for clarity
    'RLC-FT': "#0072B2",
    'Safe-DDPG': "#9D9204", 
    'Linear': '#555555'
}

# --- Visual parameters for SUBDUED individual lines ---
INDIVIDUAL_LINE_OPACITY = 1.0
INDIVIDUAL_LINE_WIDTH_FACTOR = 1.0 # Make them half of their original method_styles_base width

# --- Visual parameters for PROMINENT average lines ---
AVERAGE_LINE_OPACITY = 1.0
AVERAGE_LINE_WIDTH_INCREASE = 0.0 # Added to base method width for averages


fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Controller Output", "Cost"],
    horizontal_spacing=0.12
)
# This loop for subplot titles assumes NATURE_CONFIG is defined
for i, annotation in enumerate(fig.layout.annotations[:2]):
    annotation.font.size = NATURE_CONFIG["font_title"]
    annotation.font.family = "Arial"


# ---------------------------
# Left subplot: Controller Output (Q values) - MODIFIED for Option B
# ---------------------------
iterations_for_q_plot = np.arange(q_RL.shape[0])

# --- First, plot SUBDUED individual bus lines ---
for method in ['RLC-FT', 'Safe-DDPG', 'Linear']:
    for i, bus_data_idx in enumerate(selected_buses_indices_actual): 
        bus_display_title = selected_bus_titles_display[i]
        current_bus_color = bus_colors[i] 
        
        # Determine y_data based on method
        if method == 'RLC-FT':
            y_data = q_RL[:, bus_data_idx, 0]
        elif method == 'Safe-DDPG':
            y_data = safe_q[:, bus_data_idx, 0]
        else:  # Linear
            y_data = q_baseline[:, bus_data_idx, 0]
        
        # Get base style for the method
        base_style = method_styles_base[method]

        fig.add_trace(go.Scatter(
            x=iterations_for_q_plot,
            y=y_data,
            mode='lines',
            name=f"{bus_display_title} ({method})", 
            line=dict(
                color=current_bus_color, 
                dash=base_style['dash'],
                width=base_style['width'] * INDIVIDUAL_LINE_WIDTH_FACTOR # Subdued width
            ),
            opacity=INDIVIDUAL_LINE_OPACITY, # Subdued opacity
            legendgroup=f"Individual_{method}", 
            showlegend=True # Or False if you want to hide these from legend
        ), row=1, col=1)

# --- Second, plot PROMINENT average lines (will appear on top) ---
# for method in ['RLC-FT', 'Safe-DDPG', 'Linear']:
#     all_bus_q_data_for_method = []
#     for bus_data_idx in selected_buses_indices_actual:
#         if method == 'RLC-FT':
#             all_bus_q_data_for_method.append(q_RL[:, bus_data_idx, 0])
#         elif method == 'Safe-DDPG':
#             all_bus_q_data_for_method.append(safe_q[:, bus_data_idx, 0])
#         else:  # Linear
#             all_bus_q_data_for_method.append(q_baseline[:, bus_data_idx, 0])
    
#     if all_bus_q_data_for_method: # Check if list is not empty
#         # Stack data for mean calculation
#         stacked_q_data = np.stack(all_bus_q_data_for_method, axis=1)
#         mean_q_data = np.mean(stacked_q_data, axis=1)
        
#         base_style = method_styles_base[method]

#         fig.add_trace(go.Scatter(
#             x=iterations_for_q_plot,
#             y=mean_q_data,
#             mode='lines',
#             name=f"Average {method}",
#             line=dict(
#                 color=method_primary_colors[method], # Primary method color
#                 dash=base_style['dash'],
#                 width=base_style['width'] + AVERAGE_LINE_WIDTH_INCREASE # Prominent width
#             ),
#             opacity=AVERAGE_LINE_OPACITY, # Prominent opacity
#             legendgroup=f"Average_{method}",
#             showlegend=False 
#         ), row=1, col=1)


# ---------------------------
# Right subplot: Cost Plot - Using original method_styles_base for widths
# ---------------------------
iterations_for_cost_plot = np.arange(len(cost_RL))

fig.add_trace(go.Scatter(
    x=iterations_for_cost_plot, y=cost_RL, mode='lines', name="RLC-FT",
    line=dict(color=method_primary_colors['RLC-FT'], 
              dash=method_styles_base['RLC-FT']['dash'],
              width=method_styles_base['RLC-FT']['width']),
    legendgroup="Cost_RLCFT" # Changed legendgroup to make them distinct if needed
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=iterations_for_cost_plot, y=cost_baseline, mode='lines', name="Linear",
    line=dict(color=method_primary_colors['Linear'], 
              dash=method_styles_base['Linear']['dash'],
              width=method_styles_base['Linear']['width']),
    legendgroup="Cost_Linear"
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=iterations_for_cost_plot, y=safe_cost, mode='lines', name="Safe-DDPG",
    line=dict(color=method_primary_colors['Safe-DDPG'], 
              dash=method_styles_base['Safe-DDPG']['dash'],
              width=method_styles_base['Safe-DDPG']['width']),
    legendgroup="Cost_SafeDDPG"
), row=1, col=2)


# --- Axes and Layout (Your original settings) ---
# This part assumes NATURE_CONFIG is defined globally
fig.update_xaxes(
    title_text="Iteration Steps", range=[0, 15], 
    showgrid=True, gridwidth=0.5, gridcolor='#DDDDDD', zeroline=False, 
    tickfont=dict(size=NATURE_CONFIG["font_axis"], family="Arial"), 
    title_font=dict(size=NATURE_CONFIG["font_base"], family="Arial"),
    row=1, col=1
)
fig.update_xaxes(
    title_text="Iteration Steps", range=[0, 20], 
    showgrid=True, gridwidth=0.5, gridcolor='#DDDDDD', zeroline=False,
    tickfont=dict(size=NATURE_CONFIG["font_axis"], family="Arial"), 
    title_font=dict(size=NATURE_CONFIG["font_base"], family="Arial"),
    row=1, col=2
)
fig.update_yaxes(
    title_text="Q (MVar)", showgrid=True, gridwidth=0.5, gridcolor='#DDDDDD', zeroline=True, zerolinewidth=1, zerolinecolor='#CCCCCC',
    tickfont=dict(size=NATURE_CONFIG["font_axis"], family="Arial"), 
    title_font=dict(size=NATURE_CONFIG["font_base"], family="Arial"),
    row=1, col=1,
    # range=(0.2, 1.4)
)
fig.update_yaxes(
    title_text="Cost", showgrid=True, gridwidth=0.5, gridcolor='#DDDDDD', zeroline=False,
    tickfont=dict(size=NATURE_CONFIG["font_axis"], family="Arial"), 
    title_font=dict(size=NATURE_CONFIG["font_base"], family="Arial"),
    row=1, col=2,range=(6, 28)
)

# Subplot labels (a) and (b)
# This part assumes NATURE_CONFIG is defined globally
fig.add_annotation(
    text="<b>a</b>", x=0.01, y=1.01, xref="paper", yref="paper", showarrow=False, 
    font=dict(size=NATURE_CONFIG["font_title"]+2, family="Arial Bold", color="black"), align="left" 
)
fig.add_annotation(
    text="<b>b</b>", x=0.53, y=1.01, xref="paper", yref="paper", showarrow=False, 
    font=dict(size=NATURE_CONFIG["font_title"]+2, family="Arial Bold", color="black"), align="left"
)

# This part assumes NATURE_CONFIG is defined globally
fig.update_layout(
    font=dict(family='Arial', size=NATURE_CONFIG["font_base"]),
    width=NATURE_CONFIG["width"], 
    height=NATURE_CONFIG["height"],
    margin=dict(l=80, r=40, t=80, b=60), 
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        orientation="v", 
        yanchor="top", y=1,
        xanchor="right", x=1.20, 
        font=dict(size=NATURE_CONFIG["font_legend"], family="Arial"),
        traceorder='grouped', 
        bgcolor='rgba(255,255,255,0.0)', 
        bordercolor='lightgray', 
        borderwidth=0.5
    )
)
# Export high-resolution image
output_path = os.path.join(Config.data_path, "images","56bus", "combined_plots_nature.pdf")
fig.write_image(output_path, scale=2)
fig.write_image(os.path.join(Config.data_path, "images","56bus", "combined_plots_nature.png"), scale=2)

fig.show()